# Notebook 3 of 4 — Model Comparison, Tuning, Export

Input: `data/splits.pkl`.  Output: `model/model_pipeline.pkl`.  Next: `04_Threshold_Evaluation.ipynb`.

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, accuracy_score, recall_score,
                              precision_score, f1_score)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "model"


### Load the split from notebook 2

In [ ]:
with open(DATA_DIR / "splits.pkl", "rb") as f:
    X_train, X_test, y_train, y_test, preprocessor = pickle.load(f)

## 13. Why nine models, and why not accuracy

**The decision this section supports:** which algorithm is allowed to rank the contact list.

With a 20% churn rate, accuracy flatters a useless rule. Predicting "stay" for every customer scores about 80% and returns an empty call list. I compare nine classifiers on **F1**, which punishes both missed churners and a list full of false alarms.

**Why the imbalance treatment is inside the pipeline**

- **SMOTE** adds synthetic churn examples so the model sees the rare class often enough to learn it.
- It runs on **training data only**. The test customers stay at the real 20% rate, which is the rate the bank will face in production.
- `class_weight='balanced'` and XGBoost `scale_pos_weight` are the same idea for models that can reweight instead of resampling.

I would rather show a lower accuracy that finds leavers than a higher accuracy that finds none.

In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale = round(neg / pos, 2)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest":       RandomForestClassifier(random_state=42, class_weight="balanced"),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
    "Decision Tree":       DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "AdaBoost":            AdaBoostClassifier(random_state=42),
    "Extra Trees":         ExtraTreesClassifier(random_state=42, class_weight="balanced"),
    "Ridge Classifier":    RidgeClassifier(class_weight="balanced"),
    "SGD Classifier":      SGDClassifier(random_state=42, class_weight="balanced"),
    "XGBoost":             XGBClassifier(scale_pos_weight=scale, eval_metric="logloss", random_state=42),
}

results = []
for name, model in models.items():
    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results.append({
        "Model":     name,
        "F1":        round(f1_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
    })
    print(f"{name}: done")

pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)

## 14. Why the final model is XGBoost

Trees are the right family once EDA shows a cliff (products) and stacked risks (age × inactivity, Germany × balance). A linear model has to be told about those bends. A tree finds them.

I then tune **one** model rather than lightly tuning all nine. Randomized search (scored on F1, stratified cross-validation) spends the compute on the algorithm that already won the comparison. **SMOTETomek** is used here instead of plain SMOTE so synthetic points that land in the wrong neighbourhood are cleaned up before the tree learns them. That is a data-quality choice, not a leaderboard trick: noisy synthetic customers would become noisy call-list scores.

In [ ]:
base = ImbPipeline([
    ("preprocessor", preprocessor),
    ("resample",     SMOTETomek(random_state=42)),
    ("classifier",   XGBClassifier(eval_metric="logloss", random_state=42))
])

param_dist = {
    "classifier__n_estimators":     [200, 300, 400, 500],
    "classifier__max_depth":        [3, 4, 5, 6],
    "classifier__learning_rate":    [0.01, 0.03, 0.05, 0.1],
    "classifier__subsample":        [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "classifier__scale_pos_weight": [1, 2, 3, scale],
    "classifier__min_child_weight": [1, 3, 5],
}

search = RandomizedSearchCV(
    base, param_distributions=param_dist,
    n_iter=40, scoring="f1",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42, n_jobs=-1, verbose=1
)

search.fit(X_train, y_train)
model_pipeline = search.best_estimator_

print("Best CV F1:", round(search.best_score_, 4))
print("Best params:", search.best_params_)
print()

y_pred = model_pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"]))

## 18. Export

The saved file is the **whole pipeline** (encoding, scaling, resampling step, model), not a bare classifier. The app can send raw fields — unscaled, not pre-coded — and the same transformations used in this notebook run in production. That is the point of fitting them inside the pipeline: the score a customer gets in the demo is the score this notebook would have given them.

In [ ]:
import pickle
with open(MODEL_DIR / "model_pipeline.pkl", "wb") as f:
    pickle.dump(model_pipeline, f)